# 00 — Acquire NYC OpenData

Pulls NYPD Complaint (historic + YTD), 311 Service Requests (modern + historic),
and precinct/NTA geometries. Idempotent: parquet partitions already on disk are
skipped. Token-free runs are throttled — set `SODA_APP_TOKEN` in `.env` for speed.


In [ ]:
import os, sys
from pathlib import Path

# add src/ to path so notebooks find nyc_eda
PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT / 'src'))

from dotenv import load_dotenv
load_dotenv(PROJECT / '.env')
print('SODA token set:', bool(os.environ.get('SODA_APP_TOKEN')))


## Schema sniff — verify column names per dataset


In [ ]:
from nyc_eda import DATASETS
from nyc_eda.soda import schema_sniff

for name, info in DATASETS.items():
    df = schema_sniff(info['id'], n=2)
    print(f'{name} ({info["id"]}):  {len(df.columns)} cols')


## Geometry — precinct + NTA boundaries


In [ ]:
from nyc_eda import GEO_ROOT
from nyc_eda.geo import load_precincts, load_ntas

precincts = load_precincts(GEO_ROOT)
ntas      = load_ntas(GEO_ROOT)
print(f'precincts: {len(precincts)} polygons, columns={list(precincts.columns)[:6]}')
print(f'ntas:      {len(ntas)} polygons, columns={list(ntas.columns)[:6]}')


## Bulk pulls

Pull date ranges per dataset. Each call writes hive-partitioned parquet under
`data/raw/{dataset_id}/year=YYYY/month=MM/part.parquet` and is idempotent.

**Defaults** (override below if you want a smaller pilot):
- NYPD historic: 2017-01 to 2019-12 (the H1 lag-window pre-2020).
- NYPD YTD:      2020-01 to today.
- 311 modern:    2020-01 to today.
- 311 historic:  2017-01 to 2019-12.


In [ ]:
from datetime import date
from nyc_eda import RAW_ROOT
from nyc_eda.soda import fetch_range

today = date.today()
ranges = {
    # qgea-i56i is the canonical NYPD full-history dataset (2006-present).
    # 5uac-w243 (YTD) is reserved for the *current* calendar year only.
    'nypd_main':    (2017, 1, today.year, today.month),
    '311_modern':   (2020, 1, today.year, today.month),
    '311_historic': (2017, 1, 2019, 12),
}

all_paths = {}
for name, (sy, sm, ey, em) in ranges.items():
    info = DATASETS[name]
    all_paths[name] = fetch_range(info['id'], info['date_col'], sy, sm, ey, em, RAW_ROOT, desc=name)

for name, paths in all_paths.items():
    print(f'{name}: {len(paths)} partitions')


## DuckDB view registration


In [ ]:
from nyc_eda import DUCKDB_PATH
from nyc_eda.storage import open_db, register_dataset, register_geojson, install_spatial

con = open_db(DUCKDB_PATH)
install_spatial(con)

# per-dataset views
for name, info in DATASETS.items():
    n = register_dataset(con, f'r_{name}', RAW_ROOT, info['id'])
    print(f'r_{name}: {n:,} rows')

# unified views — match columns across historic and YTD/modern
con.execute('''
    CREATE OR REPLACE VIEW v_nypd AS
    SELECT cmplnt_num, cmplnt_fr_dt, cmplnt_fr_tm, addr_pct_cd, boro_nm,
           ofns_desc, ky_cd, law_cat_cd, latitude, longitude, prem_typ_desc,
           susp_age_group, susp_race, susp_sex, vic_age_group, vic_race, vic_sex
    FROM r_nypd_main
    UNION ALL
    SELECT cmplnt_num, cmplnt_fr_dt, cmplnt_fr_tm, addr_pct_cd, boro_nm,
           ofns_desc, ky_cd, law_cat_cd, latitude, longitude, prem_typ_desc,
           susp_age_group, susp_race, susp_sex, vic_age_group, vic_race, vic_sex
    FROM r_nypd_ytd
''')
con.execute('''
    CREATE OR REPLACE VIEW v_311 AS
    SELECT unique_key, created_date, closed_date, agency, complaint_type, descriptor,
           incident_zip, incident_address, latitude, longitude, borough, community_board,
           police_precinct, council_district, status, resolution_description
    FROM r_311_modern
    UNION ALL
    SELECT unique_key, created_date, closed_date, agency, complaint_type, descriptor,
           incident_zip, incident_address, latitude, longitude, borough, community_board,
           police_precinct, council_district, status, resolution_description
    FROM r_311_historic
''')

# geometry tables
register_geojson(con, 'g_precincts', GEO_ROOT / 'precincts.geojson')
register_geojson(con, 'g_ntas',      GEO_ROOT / 'ntas.geojson')

for v in ['v_nypd', 'v_311', 'g_precincts', 'g_ntas']:
    n = con.execute(f'SELECT COUNT(*) FROM {v}').fetchone()[0]
    print(f'{v}: {n:,} rows')


## Sanity check — date coverage per dataset


In [ ]:
for v, dcol in [('v_nypd', 'cmplnt_fr_dt'), ('v_311', 'created_date')]:
    r = con.execute(f'''
        SELECT MIN(CAST({dcol} AS TIMESTAMP)) AS min_d,
               MAX(CAST({dcol} AS TIMESTAMP)) AS max_d,
               COUNT(*) AS n
        FROM {v}
        WHERE CAST({dcol} AS TIMESTAMP) BETWEEN '2017-01-01' AND CURRENT_DATE
    ''').fetchone()
    print(f'{v}: {r[2]:,} rows from {r[0]} to {r[1]}')


## Serendipity

Anything surprising during acquisition? Schema gotchas, unexpected null patterns,
data-quality issues — log to `memos/serendipity_log.md`.
